# RiskLens: Feature Engineering & Preprocessing

## Phase 2 - Build Domain-Engineered Features & Preprocessing Pipeline

**Objective:** Create domain-relevant features and build a preprocessing pipeline that:
1. Prevents data leakage (fit preprocessor only on training data)
2. Bundles preprocessing with models for deployment
3. Handles categorical and numeric features appropriately
4. Maintains stratified class distribution across splits

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import logging

# Add src to path
sys.path.insert(0, '../')

from src.features.engineering import FeatureEngineer, prepare_data
from src.data.ingestion import load_raw_data

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries and modules imported successfully")

## 1. Load Raw Data & Create Domain Features

In [ ]:
# Load raw data
df = load_raw_data('../data/raw/data.csv')
print(f"Original shape: {df.shape}")
print(f"\nOriginal columns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")

In [ ]:
# Create domain features
fe = FeatureEngineer(random_state=42)
df_engineered = fe.create_domain_features(df)

# Show new features
new_features = [col for col in df_engineered.columns if col not in df.columns]
print(f"Created {len(new_features)} new features:")
for feat in new_features:
    print(f"  • {feat}")

print(f"\nNew shape: {df_engineered.shape}")

## 2. Explore Engineered Features

In [ ]:
# Show sample of engineered data
print("Sample of engineered features:")
print(df_engineered[['Age', 'Annual_Premium', 'Vehicle_Age', 'Vehicle_Age_Numeric', 
                      'Premium_per_Vehicle_Year', 'High_Value_Vehicle', 'Age_Risk_Bucket',
                      'Customer_Tenure_Segment', 'Premium_Bucket']].head(10))

In [ ]:
# Visualize new numeric features
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Premium per Vehicle Year', 'Premium Bucket Distribution')
)

fig.add_trace(
    go.Histogram(x=df_engineered['Premium_per_Vehicle_Year'], name='Premium/Year'),
    row=1, col=1
)

premium_bucket_counts = df_engineered['Premium_Bucket'].value_counts()
fig.add_trace(
    go.Bar(x=premium_bucket_counts.index, y=premium_bucket_counts.values, name='Count'),
    row=1, col=2
)

fig.update_layout(title_text="Engineered Numeric Features", height=400)
fig.show()

In [ ]:
# Show categorical engineered features
print("Age Risk Bucket Distribution:")
print(df_engineered['Age_Risk_Bucket'].value_counts())

print("\nCustomer Tenure Segment Distribution:")
print(df_engineered['Customer_Tenure_Segment'].value_counts())

print("\nDamage History Risk Distribution:")
print(df_engineered['Damage_History_Risk'].value_counts())

## 3. Train/Validation/Test Split with Stratification

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df_engineered.drop(['id', 'Response'], axis=1)
y = df_engineered['Response']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Overall positive rate: {y.mean():.2%}")

# First split: separate test set (15% of total)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Second split: separate train and validation (15% of remaining = 12.75% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15/(1-0.15), 
    random_state=42, stratify=y_trainval
)

print(f"\nTrain set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val set:   {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nPositive rate in each split (✓ Stratification successful):")
print(f"Overall:  {y.mean():.2%}")
print(f"Train:    {y_train.mean():.2%}")
print(f"Val:      {y_val.mean():.2%}")
print(f"Test:     {y_test.mean():.2%}")

## 4. Build Preprocessing Pipeline (Fit on Training Data ONLY)

In [ ]:
# Build and fit preprocessor ONLY on training data
fe.build_preprocessor(X_train)

print(f"✓ Preprocessor built and fitted on {len(X_train)} training samples")
print(f"\nFeature names tracked: {fe.feature_names[:5]}... ({len(fe.feature_names)} total)")

## 5. Transform All Splits (Using Fitted Preprocessor)

In [ ]:
# Transform using the fitted preprocessor
X_train_transformed = fe.transform(X_train)
X_val_transformed = fe.transform(X_val)
X_test_transformed = fe.transform(X_test)

print(f"Transformed shapes:")
print(f"  Train: {X_train_transformed.shape}")
print(f"  Val:   {X_val_transformed.shape}")
print(f"  Test:  {X_test_transformed.shape}")

print(f"\n✓ All features scaled and encoded")
print(f"Sample transformed features (first 5 samples, first 5 features):")
print(X_train_transformed[:5, :5])

## 6. Save Processed Data & Preprocessor

In [ ]:
import os
from pathlib import Path

output_dir = '../data/processed'
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Save as numpy
np.save(f'{output_dir}/X_train.npy', X_train_transformed)
np.save(f'{output_dir}/X_val.npy', X_val_transformed)
np.save(f'{output_dir}/X_test.npy', X_test_transformed)
np.save(f'{output_dir}/y_train.npy', y_train.values)
np.save(f'{output_dir}/y_val.npy', y_val.values)
np.save(f'{output_dir}/y_test.npy', y_test.values)

# Save as pickle for pandas compatibility
pd.DataFrame(X_train_transformed).to_pickle(f'{output_dir}/X_train.pkl')
pd.DataFrame(X_val_transformed).to_pickle(f'{output_dir}/X_val.pkl')
pd.DataFrame(X_test_transformed).to_pickle(f'{output_dir}/X_test.pkl')
y_train.to_pickle(f'{output_dir}/y_train.pkl')
y_val.to_pickle(f'{output_dir}/y_val.pkl')
y_test.to_pickle(f'{output_dir}/y_test.pkl')

# Save preprocessor
fe.save_preprocessor()

print(f"✓ All processed data saved to {output_dir}")
print(f"✓ Preprocessor saved to artifacts/preprocessor.pkl")

## 7. Feature Engineering Summary

In [ ]:
print("="*70)
print("PHASE 2 - FEATURE ENGINEERING SUMMARY")
print("="*70)

print("\n1. DOMAIN FEATURES CREATED:")
new_features = [
    'Vehicle_Age_Numeric - Numeric vehicle age for calculations',
    'Premium_per_Vehicle_Year - Exposure-adjusted premium (risk/year)',
    'High_Value_Vehicle - Flag for premium > 75th percentile',
    'Age_Risk_Bucket - Insurance risk category based on age',
    'Customer_Tenure_Segment - Customer lifecycle stage',
    'Premium_Bucket - Premium amount category',
    'Damage_History_Risk - Interaction: damage × not previously insured'
]
for i, feat in enumerate(new_features, 1):
    print(f"   {i}. {feat}")

print(f"\n2. DATA SPLITS (with Stratification):")
print(f"   Train: {X_train.shape[0]:>8} samples ({X_train.shape[0]/len(X)*100:>5.1f}%) | Positive: {y_train.mean():>6.2%}")
print(f"   Val:   {X_val.shape[0]:>8} samples ({X_val.shape[0]/len(X)*100:>5.1f}%) | Positive: {y_val.mean():>6.2%}")
print(f"   Test:  {X_test.shape[0]:>8} samples ({X_test.shape[0]/len(X)*100:>5.1f}%) | Positive: {y_test.mean():>6.2%}")

print(f"\n3. PREPROCESSING PIPELINE:")
print(f"   ✓ Fitted ONLY on training data (prevents data leakage)")
print(f"   ✓ StandardScaler for numeric features")
print(f"   ✓ OrdinalEncoder for categorical features")
print(f"   ✓ Handles unknown categorical values with -1")
print(f"   ✓ Median imputation for numeric nulls")
print(f"   ✓ Mode imputation for categorical nulls")

print(f"\n4. KEY ADVANTAGES:")
print(f"   ✓ No leakage: Preprocessor fitted only on training data")
print(f"   ✓ Deployment-ready: Bundled pipeline for production")
print(f"   ✓ Reproducible: All transformations tracked and saved")
print(f"   ✓ Scalable: Works with any new data using fitted statistics")

print(f"\n5. READY FOR PHASE 3:")
print(f"   ✓ Processed data splits saved")
print(f"   ✓ Preprocessor saved for future use")
print(f"   ✓ Next: Baseline modeling (Logistic Regression + Decision Tree)")

print("\n" + "="*70)